# Tesseract Maltese Fine-Tuning (Tesstrain-Style, FineTuning Local Data)

This notebook builds a reproducible Tesseract LSTM fine-tuning pipeline using only suthetic data.

It follows the workflow style from:
- https://github.com/Matleo/Tesseract_fine_tuning_training

## 0. Input Parameters

In [ ]:
from __future__ import annotations

import json
import os
import random
import re
import shutil
import subprocess
import unicodedata
import urllib.request
from dataclasses import dataclass
from pathlib import Path
from typing import Iterable

import numpy as np
import pandas as pd
import pytesseract
from PIL import Image


@dataclass
class RunConfig:
    include_languages: tuple[str, ...] = ("Maltese", "Mixed")
    random_seed: int = 42
    eval_page_fraction: float = 0.10 # Fraction of pages to use for evaluation (rest used for training)
    max_samples: int | None = None  # set int for smoke runs
    model_name: str = "mlt_custom_v1"
    base_lang_candidates: tuple[str, ...] = ("mlt", "eng")
    max_iterations: int = 500  
    target_error_rate: float | None = None 
    learning_rate: float = 1e-2 
    perfect_sample_delay: int = 10  
    debug_interval: int = 50 
    dehyphenate_predictions: bool = True
    prefer_tessdata_best: bool = True


CFG = RunConfig()
print(CFG)

In [ ]:
# Resolve paths robustly even if the notebook kernel starts in an unexpected cwd.
DATASET_DIR_CANDIDATES = (
    "32k_114138_synthData",
    "32k_synthData",
)


def _is_finetuning_root(p: Path) -> bool:
    return any((p / name).exists() for name in DATASET_DIR_CANDIDATES)


def _find_finetuning_root(start: Path) -> Path | None:
    candidates = []

    # Allow explicit override in shell/session for fully custom layouts.
    env_root = os.environ.get("NOMOCRAT_FINETUNING_ROOT")
    if env_root:
        candidates.append(Path(env_root))

    # Strong preference: current directory if it is FineTuning-like.
    candidates.append(start)

    # Prefer explicit FineTuning subpaths before generic parent roots.
    for parent in [start, *start.parents]:
        candidates.append(parent / "FineTuning")

    # Then try parent chain itself.
    candidates.extend(start.parents)

    seen = set()
    for c in candidates:
        c = c.resolve()
        if c in seen:
            continue
        seen.add(c)
        if _is_finetuning_root(c):
            return c
    return None


cwd = Path.cwd().resolve()
FT_ROOT = _find_finetuning_root(cwd)
if FT_ROOT is None:
    raise RuntimeError(
        "Could not resolve FineTuning root. "
        f"Expected a folder containing one of {DATASET_DIR_CANDIDATES}. "
        "Optionally set NOMOCRAT_FINETUNING_ROOT to an explicit path. "
        f"Current cwd is: {cwd}"
    )

LOCAL_DATA = next(
    (FT_ROOT / name for name in DATASET_DIR_CANDIDATES if (FT_ROOT / name).exists()),
    FT_ROOT / DATASET_DIR_CANDIDATES[0],
)
# TRAIN_JSON = LOCAL_DATA / CFG.train_json_name
# EVAL_JSON = LOCAL_DATA / CFG.eval_json_name
# CROPS_DEV = LOCAL_DATA / "train" / "images"
# CROPS_TEST = LOCAL_DATA / "test" / "images"
GT_TXT_PATH = LOCAL_DATA / "train" / "gt.txt"
IMAGES_DIR = LOCAL_DATA / "train" / "images"

TESSTRAIN_ROOT = FT_ROOT / "Testing" / "tesstrain_data"
LINE_PAIRS_DIR = TESSTRAIN_ROOT / "line_pairs"
LSTMF_DIR = TESSTRAIN_ROOT / "lstmf"
LISTS_DIR = TESSTRAIN_ROOT / "lists"
MODEL_DIR = TESSTRAIN_ROOT / "model"
LOGS_DIR = TESSTRAIN_ROOT / "logs"
TESSDATA_CUSTOM = TESSTRAIN_ROOT / "tessdata_custom"

for p in [TESSTRAIN_ROOT, LINE_PAIRS_DIR, LSTMF_DIR, LISTS_DIR, MODEL_DIR, LOGS_DIR, TESSDATA_CUSTOM]:
    p.mkdir(parents=True, exist_ok=True)

print(f"Kernel cwd:      {cwd}")
print(f"FineTuning root: {FT_ROOT}")
print(f"Local data:      {LOCAL_DATA}")
print(f"GT path:         {GT_TXT_PATH}")
print(f"Images dir:      {IMAGES_DIR}")
print(f"Artifacts:       {TESSTRAIN_ROOT}")

## 1. Environment Checks

This section verifies required binaries and locates base `.traineddata` files.

Note: Tesseract LSTM training is CPU-based. Multiple GPUs on this machine will not accelerate `lstmtraining` directly.

In [ ]:
def run_cmd(cmd: list[str], check: bool = True, env: dict[str, str] | None = None) -> subprocess.CompletedProcess:
    return subprocess.run(cmd, check=check, capture_output=True, text=True, env=env)


def command_exists(cmd: str) -> bool:
    return shutil.which(cmd) is not None


required_bins = ["tesseract", "combine_tessdata", "lstmtraining", "lstmeval"]
missing = [b for b in required_bins if not command_exists(b)]

print("Binary lookup:")
for b in required_bins:
    print(f"- {b}: {shutil.which(b)}")

if missing:
    install_hint = (
        "Missing required Tesseract training binaries: " + ", ".join(missing) + "\n\n"
        "Your current conda env has only the runtime OCR binary.\n"
        "Fine-tuning requires Tesseract training tools (combine_tessdata/lstmtraining/lstmeval).\n\n"
        "Install option A (recommended on Ubuntu):\n"
        "  sudo apt update\n"
        "  sudo apt install tesseract-ocr libtesseract-dev tesseract-ocr-eng\n\n"
        "Then verify:\n"
        "  which combine_tessdata && which lstmtraining && which lstmeval\n\n"
        "Install option B:\n"
        "  Build Tesseract from source with training tools enabled."
    )
    raise RuntimeError(install_hint)

print("\nVersion info:")
print(run_cmd(["tesseract", "--version"]).stdout.splitlines()[0])
print(run_cmd(["combine_tessdata", "--help"], check=False).stdout.splitlines()[0])
print(run_cmd(["lstmtraining", "--help"], check=False).stdout.splitlines()[0])
print(run_cmd(["lstmeval", "--help"], check=False).stdout.splitlines()[0])

In [ ]:
def discover_tessdata_dirs() -> list[Path]:
    env_dirs: list[Path] = []
    tessdata_prefix = os.environ.get("TESSDATA_PREFIX")
    if tessdata_prefix:
        p = Path(tessdata_prefix)
        if p.name == "tessdata":
            env_dirs.append(p)
        else:
            env_dirs.append(p / "tessdata")
            env_dirs.append(p)

    candidates = [
        FT_ROOT / "tessdata",
        Path("/usr/share/tesseract-ocr/5/tessdata"),
        Path("/usr/share/tesseract-ocr/4.00/tessdata"),
        Path("/usr/share/tesseract-ocr/tessdata"),
        Path("/usr/local/share/tessdata"),
    ]

    ordered = []
    seen = set()
    for d in env_dirs + candidates:
        if d not in seen and d.exists():
            ordered.append(d)
            seen.add(d)
    return ordered


def find_traineddata(lang: str, search_dirs: Iterable[Path]) -> Path | None:
    for d in search_dirs:
        td = d / f"{lang}.traineddata"
        if td.exists():
            return td
    return None


TESSDATA_DIRS = discover_tessdata_dirs()
print("Candidate tessdata dirs:")
for d in TESSDATA_DIRS:
    print(f"- {d}")

BASE_TRAINEDDATA_PATH = None
BASE_LANG_USED = None
for lang in CFG.base_lang_candidates:
    candidate = find_traineddata(lang, TESSDATA_DIRS)
    if candidate is not None:
        BASE_TRAINEDDATA_PATH = candidate
        BASE_LANG_USED = lang
        break

if BASE_TRAINEDDATA_PATH is None:
    raise RuntimeError(
        f"Could not find any base traineddata among candidates {CFG.base_lang_candidates}. "
        "Install mlt.traineddata (preferred) or eng.traineddata in a tessdata dir."
    )

print(f"Base traineddata: {BASE_TRAINEDDATA_PATH}")
print(f"Base lang used:  {BASE_LANG_USED}")

## 2. Build Training Index From FineTuning Data

In [ ]:
# --- Build manifest from .gt.txt and images directly ---
from pathlib import Path
import pandas as pd


rows = []
with open(GT_TXT_PATH, encoding='utf-8') as f:
    for line in f:
        line = line.strip()
        if not line:
            continue
        # Typical format: filename.tif (or .jpg)\tlabel
        if '\t' in line:
            fname, gt_text = line.split('\t', 1)
        elif ' ' in line:
            fname, gt_text = line.split(' ', 1)
        else:
            continue  # skip malformed lines
        img_path = IMAGES_DIR / fname
        if not img_path.exists():
            print(f'Warning: image not found: {img_path}')
            continue
        rows.append({
            'sample_id': Path(fname).stem,
            'page_fname': fname,
            'gt_text': gt_text,
            'crop_path': str(img_path),
            'exists': img_path.exists(),
            'split': 'train',  # or set up your own split logic
            'box_idx': 0,
            'language': 'Maltese',  # or set as needed
        })

manifest = pd.DataFrame(rows)
print(f'Loaded {len(manifest)} samples from gt.txt')
manifest.head()

In [ ]:
print(f"Number of rows constructed: {len(rows)}")
if len(rows) > 0:
    print("First row:", rows[0])
print("Manifest DataFrame head:")
print(manifest.head())

## 3. Create Tesstrain Line Pairs (`.tif`, `.gt.txt`, `.box`)

Use tesstrain-compatible **line-level** `.box` files generated from the ground truth text. This keeps the workflow standard while avoiding the earlier fragile hand-made box format.

In [ ]:
def safe_text(v):
    if v is None:
        return ""
    s = str(v)
    import unicodedata, re
    s = unicodedata.normalize("NFC", s)
    s = re.sub(r"\s+", " ", s).strip()
    return s

In [ ]:
# Filter training rows
lang_set = set(CFG.include_languages)
df_filtered = manifest.copy()
df_filtered = df_filtered[df_filtered["exists"]]
df_filtered = df_filtered[df_filtered["gt_text"].map(lambda s: safe_text(s) != "")]
df_filtered = df_filtered[df_filtered["language"].isin(lang_set)]

if CFG.max_samples is not None:
    df_filtered = df_filtered.head(CFG.max_samples)

if df_filtered.empty:
    raise RuntimeError("No training rows after filtering. Adjust include_languages or verify data.")

print(f"Rows used for tesstrain pair generation: {len(df_filtered)}")
print(df_filtered["language"].value_counts(dropna=False))

# Stable split by page to reduce leakage between train/eval
pages = sorted(df_filtered["page_fname"].dropna().unique().tolist())
rng = random.Random(CFG.random_seed)
rng.shuffle(pages)

n_eval_pages = max(1, int(len(pages) * CFG.eval_page_fraction))
eval_pages = set(pages[:n_eval_pages])

manifest_rows = []
for i, row in enumerate(df_filtered.itertuples(index=False), start=1):
    split = "eval" if row.page_fname in eval_pages else "train"
    sample_id = f"sample_{i:07d}"
    manifest_rows.append(
        {
            "sample_id": sample_id,
            "split": split,
            "page_fname": row.page_fname,
            "box_idx": int(row.box_idx),
            "language": row.language,
            "gt_text": safe_text(row.gt_text),
            "crop_path": row.crop_path,
        }
    )

manifest = pd.DataFrame(manifest_rows)
print(manifest["split"].value_counts())
manifest.head(10)

In [ ]:
# Debug: Check columns before filtering by 'language'
print('df_filtered columns:', df_filtered.columns.tolist())
print('First few rows:')
print(df_filtered.head())

In [ ]:
# Create tesstrain-style line pairs from the cropped images.
# Use standard line-level `.box` files (matching tesstrain's `generate_line_box.py`)
# together with adjacent `.gt.txt` files for LSTM training.

def write_line_box_file(box_path: Path, text: str, width: int, height: int) -> None:
    line = unicodedata.normalize("NFC", text).strip()
    with box_path.open("w", encoding="utf-8") as f:
        if not line:
            return

        for i in range(1, len(line)):
            char = line[i]
            prev_char = line[i - 1]
            if unicodedata.combining(char):
                f.write(f"{prev_char + char} 0 0 {width} {height} 0\n")
            elif not unicodedata.combining(prev_char):
                f.write(f"{prev_char} 0 0 {width} {height} 0\n")

        if not unicodedata.combining(line[-1]):
            f.write(f"{line[-1]} 0 0 {width} {height} 0\n")

        # Required end-of-textline marker for line-box training data.
        f.write(f"\t 0 0 {width} {height} 0\n")


def save_tif_and_labels(sample: pd.Series, out_dir: Path) -> dict[str, str]:
    sample_id = str(sample["sample_id"])
    img_in = Path(sample["crop_path"])
    tif_path = out_dir / f"{sample_id}.tif"
    gt_path = out_dir / f"{sample_id}.gt.txt"
    box_path = out_dir / f"{sample_id}.box"

    text = safe_text(sample["gt_text"])
    with Image.open(img_in) as im:
        # Keep a simple grayscale rendering for robust tesstrain line processing.
        gray = im.convert("L")
        gray.save(tif_path, format="TIFF", dpi=(300, 300))
        w, h = gray.size

    gt_path.write_text(text + "\n", encoding="utf-8")
    write_line_box_file(box_path, text, w, h)

    return {
        "sample_id": sample_id,
        "split": str(sample["split"]),
        "page_fname": str(sample["page_fname"]),
        "box_idx": str(sample["box_idx"]),
        "tif_path": str(tif_path),
        "gt_path": str(gt_path),
        "box_path": str(box_path),
    }


# Recreate line-pair folder for clean reproducibility.
if LINE_PAIRS_DIR.exists():
    shutil.rmtree(LINE_PAIRS_DIR)
LINE_PAIRS_DIR.mkdir(parents=True, exist_ok=True)

written = [save_tif_and_labels(row, LINE_PAIRS_DIR) for _, row in manifest.iterrows()]
df_pairs = pd.DataFrame(written)

chars = "".join(manifest["gt_text"].tolist())
char_set = sorted(set(chars))
interesting = [c for c in ["ċ", "Ċ", "ġ", "Ġ", "għ", "ħ", "Ħ", "ż", "Ż", "à", "è", "ì", "ò", "ù"] if c in "".join(char_set)]

print(f"Generated tif/gt/box triples: {len(df_pairs)}")
print(f"Unique characters: {len(char_set)}")
print(f"Detected key Maltese/accent tokens in dataset: {interesting}")
print("Using tesstrain-compatible line-level box files with an end-of-line marker.")

df_pairs.head(8)

## 4. Generate `.lstmf` Files And Split Lists

In [ ]:
# Disable ScrollView GUI to prevent training from hanging on headless systems
CFG.debug_interval = 0
print('Set CFG.debug_interval =', CFG.debug_interval)

In [ ]:
if LSTMF_DIR.exists():
    shutil.rmtree(LSTMF_DIR)
LSTMF_DIR.mkdir(parents=True, exist_ok=True)

lstm_tessdata_dir = str(BASE_TRAINEDDATA_PATH.parent.resolve())
lstm_train_config = None
for d in TESSDATA_DIRS:
    candidate = d / 'configs' / 'lstm.train'
    if candidate.exists():
        lstm_train_config = str(candidate.resolve())
        break

print(f'Using tessdata dir for lstmf generation: {lstm_tessdata_dir}')
print(f'Using lstm.train config: {lstm_train_config or "lstm.train"}')

failed = []
for tif_path in sorted(LINE_PAIRS_DIR.glob("*.tif")):
    stem = tif_path.stem
    outbase = LSTMF_DIR / stem
    gt_path = LINE_PAIRS_DIR / f"{stem}.gt.txt"

    if not gt_path.exists():
        failed.append(
            {
                "sample_file": tif_path.name,
                "stderr": f"Missing ground-truth file: {gt_path.name}",
            }
        )
        continue

    cmd = [
        "tesseract",
        str(tif_path),
        str(outbase),
        "--tessdata-dir", lstm_tessdata_dir,
        "--psm", "13",
        "-l", BASE_LANG_USED,
        lstm_train_config or "lstm.train",
    ]
    proc = run_cmd(cmd, check=False)
    lstmf_path = LSTMF_DIR / f"{stem}.lstmf"
    if proc.returncode != 0 or not lstmf_path.exists():
        failed.append(
            {
                "sample_file": tif_path.name,
                "stderr": (proc.stderr or proc.stdout)[-1000:].strip() or "No .lstmf output generated.",
            }
        )

lstmf_files = sorted(str(p.resolve()) for p in LSTMF_DIR.glob("*.lstmf"))
if not lstmf_files:
    raise RuntimeError(
        "No .lstmf files generated. Check whether the line images and `.gt.txt` pairs are valid."
    )

if failed:
    LOGS_DIR.mkdir(parents=True, exist_ok=True)
    failed_df = pd.DataFrame(failed)
    failed_path = LOGS_DIR / "lstmf_failures.tsv"
    failed_df.to_csv(failed_path, sep="\t", index=False)
    print(f"Warning: failed lstmf generations: {len(failed)}")
    print(f"Failure log saved to: {failed_path}")
    print(failed_df.head(10))

pair_split = {row.sample_id: row.split for row in df_pairs.itertuples(index=False)}
train_list = []
eval_list = []
for p in lstmf_files:
    sid = Path(p).stem
    if pair_split.get(sid, "train") == "eval":
        eval_list.append(p)
    else:
        train_list.append(p)

LISTS_DIR.mkdir(parents=True, exist_ok=True)
all_file = LISTS_DIR / "all-lstmf.txt"
train_file = LISTS_DIR / "list_train.txt"
eval_file = LISTS_DIR / "list_eval.txt"

all_file.write_text("\n".join(lstmf_files) + "\n", encoding="utf-8")
train_file.write_text("\n".join(train_list) + "\n", encoding="utf-8")
eval_file.write_text("\n".join(eval_list) + "\n", encoding="utf-8")

print(f"Total lstmf: {len(lstmf_files)}")
print(f"Train list:  {len(train_list)}")
print(f"Eval list:   {len(eval_list)}")
print(f"Failures:    {len(failed)}")
print(f"Saved lists in: {LISTS_DIR}")
print("Used Tesseract's native `lstm.train` flow with `.gt.txt` files instead of synthetic `.box` files.")

In [ ]:
# Optional quick smoke test for the tesstrain-compatible line-box path.
# Run this before a full `.lstmf` generation pass if you want a fast sanity check.

SMOKE_COUNT = 25
smoke_out_dir = LOGS_DIR / 'lstmf_smoke'
if smoke_out_dir.exists():
    shutil.rmtree(smoke_out_dir)
smoke_out_dir.mkdir(parents=True, exist_ok=True)

lstm_tessdata_dir = str(BASE_TRAINEDDATA_PATH.parent.resolve())
lstm_train_config = None
for d in TESSDATA_DIRS:
    candidate = d / 'configs' / 'lstm.train'
    if candidate.exists():
        lstm_train_config = str(candidate.resolve())
        break

smoke_pairs = sorted(LINE_PAIRS_DIR.glob('*.tif'))[:SMOKE_COUNT]
smoke_results = []

for tif_path in smoke_pairs:
    stem = tif_path.stem
    outbase = smoke_out_dir / stem
    cmd = [
        'tesseract',
        str(tif_path),
        str(outbase),
        '--tessdata-dir', lstm_tessdata_dir,
        '--psm', '13',
        '-l', BASE_LANG_USED,
        lstm_train_config or 'lstm.train',
    ]
    proc = run_cmd(cmd, check=False)
    lstmf_path = smoke_out_dir / f'{stem}.lstmf'
    smoke_results.append(
        {
            'sample_file': tif_path.name,
            'ok': proc.returncode == 0 and lstmf_path.exists(),
            'returncode': proc.returncode,
            'stderr_tail': (proc.stderr or proc.stdout)[-240:].strip(),
        }
    )

smoke_df = pd.DataFrame(smoke_results)
print(f"Using tessdata dir: {lstm_tessdata_dir}")
print(f"Using lstm.train config: {lstm_train_config or 'lstm.train'}")
print(f"Smoke test success: {int(smoke_df['ok'].sum())}/{len(smoke_df)}")
display(smoke_df.head(10))

## 5. Extract Base LSTM, Train, Evaluate, Package

In [ ]:
import time

BASE_LSTM_PATH = MODEL_DIR / f"{BASE_LANG_USED}.lstm"
MODEL_PREFIX = MODEL_DIR / CFG.model_name
TRAIN_LOG = LOGS_DIR / f"{CFG.model_name}_train.log"
EVAL_LOG = LOGS_DIR / f"{CFG.model_name}_eval.log"


def ensure_trainable_base(lang: str, traineddata_path: Path) -> tuple[Path, Path]:
    """Returns (trainable_lstm_path, traineddata_path)."""
    # Try user-discovered traineddata first.
    lstm_path = MODEL_DIR / f"{lang}.lstm"
    extract_cmd = ["combine_tessdata", "-e", str(traineddata_path), str(lstm_path)]
    extract_proc = run_cmd(extract_cmd, check=False)
    if extract_proc.returncode != 0:
        raise RuntimeError(f"combine_tessdata extraction failed:\n{extract_proc.stderr}")

    if not CFG.prefer_tessdata_best:
        return lstm_path, traineddata_path

    # Prefer float model from tessdata_best to avoid integer-model training failures.
    best_dir = TESSDATA_CUSTOM / "base_tessdata"
    best_dir.mkdir(parents=True, exist_ok=True)
    best_td = best_dir / f"{lang}.traineddata"
    best_lstm = MODEL_DIR / f"{lang}_best.lstm"

    if not best_td.exists():
        url = f"https://github.com/tesseract-ocr/tessdata_best/raw/main/{lang}.traineddata"
        print(f"Downloading tessdata_best base model: {url}")
        urllib.request.urlretrieve(url, best_td)

    best_extract_cmd = ["combine_tessdata", "-e", str(best_td), str(best_lstm)]
    best_extract_proc = run_cmd(best_extract_cmd, check=False)
    if best_extract_proc.returncode == 0 and best_lstm.exists():
        print(f"Using tessdata_best base model: {best_td}")
        return best_lstm, best_td

    print("Warning: Could not use tessdata_best base model, falling back to discovered model.")
    return lstm_path, traineddata_path


def format_duration(seconds: float) -> str:
    total_seconds = int(round(seconds))
    hours, remainder = divmod(total_seconds, 3600)
    minutes, seconds = divmod(remainder, 60)
    return f"{hours}h {minutes}m {seconds}s"


# 1) Prepare trainable base network
BASE_LSTM_PATH, BASE_TRAINEDDATA_PATH = ensure_trainable_base(BASE_LANG_USED, BASE_TRAINEDDATA_PATH)
print(f"Base LSTM extracted: {BASE_LSTM_PATH}")
print(f"Base traineddata:    {BASE_TRAINEDDATA_PATH}")

# 2) Run training
train_cmd = [
    "lstmtraining",
    "--model_output", str(MODEL_PREFIX),
    "--continue_from", str(BASE_LSTM_PATH),
    "--traineddata", str(BASE_TRAINEDDATA_PATH),
    "--train_listfile", str(LISTS_DIR / "list_train.txt"),
    "--eval_listfile", str(LISTS_DIR / "list_eval.txt"),
    "--max_iterations", str(CFG.max_iterations),
    "--learning_rate", str(CFG.learning_rate),
    # "--perfect_sample_delay", str(CFG.perfect_sample_delay),
    # "--debug_interval", str(CFG.debug_interval),
]
if CFG.target_error_rate is not None:
    train_cmd.extend(["--target_error_rate", str(CFG.target_error_rate)])

print("Training hyperparameters:")
print(f"- max_iterations:        {CFG.max_iterations}")
if CFG.target_error_rate is None:
    print("- target_error_rate:     disabled")
else:
    print(f"- target_error_rate:     {CFG.target_error_rate}")
print(f"- learning_rate:        {CFG.learning_rate}")
# print(f"- perfect_sample_delay: {CFG.perfect_sample_delay}")

training_started_at = pd.Timestamp.now()
training_started_perf = time.perf_counter()
with TRAIN_LOG.open("w", encoding="utf-8") as logf:
    train_proc = subprocess.run(train_cmd, text=True, stdout=logf, stderr=subprocess.STDOUT)
training_elapsed_seconds = time.perf_counter() - training_started_perf
training_finished_at = pd.Timestamp.now()
training_runtime = format_duration(training_elapsed_seconds)

print(f"Training return code: {train_proc.returncode}")
print(f"Training log: {TRAIN_LOG}")
print(f"Training started: {training_started_at}")
print(f"Training finished: {training_finished_at}")
print(f"Training elapsed: {training_runtime}")
if train_proc.returncode != 0:
    raise RuntimeError(
        "Training failed. Check the train log at "
        f"{TRAIN_LOG}"
    )

# 3) Evaluate latest checkpoint if available
checkpoints = sorted(MODEL_DIR.glob(f"{CFG.model_name}_checkpoint"))
if checkpoints:
    checkpoint = checkpoints[-1]
    eval_cmd = [
        "lstmeval",
        "--model", str(checkpoint),
        "--traineddata", str(BASE_TRAINEDDATA_PATH),
        "--eval_listfile", str(LISTS_DIR / "list_eval.txt"),
    ]
    with EVAL_LOG.open("w", encoding="utf-8") as logf:
        eval_proc = subprocess.run(eval_cmd, text=True, stdout=logf, stderr=subprocess.STDOUT)
    print(f"Eval return code: {eval_proc.returncode}")
    print(f"Eval log: {EVAL_LOG}")
else:
    print("No checkpoint found for lstmeval yet; this can happen for very short runs.")

In [ ]:
# Package final traineddata from the newest checkpoint
MODEL_TRAINEDDATA = TESSDATA_CUSTOM / f"{CFG.model_name}.traineddata"

checkpoint_candidates = sorted(MODEL_DIR.glob(f"{CFG.model_name}_checkpoint*"))
if not checkpoint_candidates:
    raise RuntimeError(
        "No checkpoint found to package. "
        "Run the training cell first and confirm it exits with return code 0."
    )

# Pick the most recently modified checkpoint-like artifact.
final_checkpoint = max(checkpoint_candidates, key=lambda p: p.stat().st_mtime)
package_cmd = [
    "lstmtraining",
    "--stop_training",
    "--continue_from", str(final_checkpoint),
    "--traineddata", str(BASE_TRAINEDDATA_PATH),
    "--model_output", str(MODEL_TRAINEDDATA),
]
pack_proc = run_cmd(package_cmd, check=False)
if pack_proc.returncode != 0:
    raise RuntimeError(f"Packaging failed:\n{pack_proc.stderr}")

print(f"Packaged model: {MODEL_TRAINEDDATA}")
print(f"Checkpoint used: {final_checkpoint}")
print("Use language code for inference:", CFG.model_name)

In [ ]:
# Estimate elapsed training time for an already completed run from checkpoint timestamps
def _format_duration(seconds: float) -> str:
    total_seconds = int(round(seconds))
    hours, remainder = divmod(total_seconds, 3600)
    minutes, seconds = divmod(remainder, 60)
    return f'{hours}h {minutes}m {seconds}s'

checkpoint_files = sorted(MODEL_DIR.glob(f'{CFG.model_name}_*.checkpoint'), key=lambda p: p.stat().st_mtime)

if checkpoint_files:
    checkpoint_start = pd.Timestamp(checkpoint_files[0].stat().st_mtime, unit='s')
    checkpoint_end = pd.Timestamp(checkpoint_files[-1].stat().st_mtime, unit='s')
    checkpoint_elapsed_seconds = checkpoint_files[-1].stat().st_mtime - checkpoint_files[0].stat().st_mtime
    print(f'Estimated training start from checkpoints: {checkpoint_start}')
    print(f'Estimated training end from checkpoints:   {checkpoint_end}')
    print(f'Estimated elapsed time: {_format_duration(checkpoint_elapsed_seconds)}')
else:
    print('No checkpoint history found to estimate training time.')

## 6. Baseline `mlt` vs Fine-Tuned Model on JSON or `gt.txt`

This section supports both the original **Levenshtein-based CER** comparison on the `v11` JSON format and direct evaluation from a line-level `gt.txt` file paired with cropped images.

In [ ]:
# Apples-to-apples evaluation on either:
# 1) the v11 JSON structure used in `Evaluation/Tesseract/tesseract_mlt_test.ipynb`, or
# 2) a simple `gt.txt` file with `image_name<TAB>transcription` lines.
#
# Run the baseline `mlt` and the fine-tuned model on the exact same matched crops,
# then compare CER side by side.
import json
import unicodedata
from pathlib import Path

import pandas as pd
import pytesseract
from PIL import Image


def normalize_text(s: str) -> str:
    s = unicodedata.normalize("NFC", str(s))
    s = " ".join(s.split())
    return s


def normalize_eval_text(value):
    if value is None:
        return ""
    text = normalize_text(value)
    if 'safe_text' in globals():
        text = normalize_text(safe_text(text))
    return text


def levenshtein_distance(a: str, b: str) -> int:
    if a == b:
        return 0
    if len(a) == 0:
        return len(b)
    if len(b) == 0:
        return len(a)

    prev = list(range(len(b) + 1))
    for i, ca in enumerate(a, start=1):
        curr = [i]
        for j, cb in enumerate(b, start=1):
            ins = curr[j - 1] + 1
            dele = prev[j] + 1
            sub = prev[j - 1] + (ca != cb)
            curr.append(min(ins, dele, sub))
        prev = curr
    return prev[-1]


def ocr_with_model(image: Image.Image, lang_code: str, tessdata_dir: Path | None = None) -> str:
    config_parts = []
    if tessdata_dir is not None:
        config_parts.append(f'--tessdata-dir "{tessdata_dir.resolve()}"')
    config = ' '.join(config_parts).strip()
    return pytesseract.image_to_string(image, lang=lang_code, config=config)


def load_eval_samples(test_source: Path, crops_dir: Path):
    test_source = Path(test_source)
    crops_dir = Path(crops_dir)

    all_images = []
    for pattern in ('*.jpg', '*.png', '*.tif'):
        all_images.extend(crops_dir.rglob(pattern))

    image_map = {img_path.relative_to(crops_dir).as_posix(): img_path for img_path in all_images}
    image_name_map = {}
    for img_path in all_images:
        image_name_map.setdefault(img_path.name, img_path)

    samples = []
    missing_paths = []
    records = []
    suffix = test_source.suffix.lower()

    if suffix == '.json':
        with open(test_source, encoding='utf-8') as f:
            raw_data = json.load(f)

        records = raw_data.get('data', []) if isinstance(raw_data, dict) else raw_data
        if not isinstance(records, list):
            raise RuntimeError(f'Unsupported test JSON structure: {type(raw_data)}')

        for rec in records:
            if not isinstance(rec, dict):
                continue

            page = rec.get('page', {}) if isinstance(rec.get('page'), dict) else {}
            page_fname = str(page.get('page_fname') or rec.get('page_fname') or '').strip()
            document_id = str(page.get('document_id') or '').strip()
            page_id = str(page.get('page_id') or '').strip()
            stem = Path(page_fname).stem if page_fname else ''
            boxes = rec.get('boxes', []) if isinstance(rec.get('boxes'), list) else []

            if not stem or not document_id or not page_id or not boxes:
                continue

            for box_idx, box in enumerate(boxes):
                if not isinstance(box, dict):
                    continue

                gt_text = normalize_eval_text(box.get('transcription', ''))
                if not gt_text:
                    continue

                expected_rel_paths = [
                    f'{document_id}/{page_id}/{stem}_box{box_idx:04d}.jpg',
                    f'{document_id}/{page_id}/{stem}_box{box_idx:04d}.png',
                    f'{document_id}/{page_id}/{stem}_box{box_idx:04d}.tif',
                ]

                found_path = None
                found_rel_path = None
                for rel_path in expected_rel_paths:
                    candidate = image_map.get(rel_path)
                    if candidate is not None:
                        found_path = candidate
                        found_rel_path = rel_path
                        break

                if found_path is None:
                    if len(missing_paths) < 10:
                        missing_paths.append(expected_rel_paths[0])
                    continue

                samples.append({
                    'page_fname': page_fname or f'{document_id}-{page_id}.jpg',
                    'image': str(found_path),
                    'relative_image': found_rel_path,
                    'document_id': document_id,
                    'page_id': page_id,
                    'box_idx': box_idx,
                    'gt_text': gt_text,
                })

        dataset_name = test_source.stem

    elif suffix == '.txt':
        with open(test_source, encoding='utf-8') as f:
            for line_no, line in enumerate(f, start=1):
                line = line.strip()
                if not line:
                    continue

                if '\t' in line:
                    fname, gt_text = line.split('\t', 1)
                elif ' ' in line:
                    fname, gt_text = line.split(' ', 1)
                else:
                    continue

                gt_text = normalize_eval_text(gt_text)
                if not gt_text:
                    continue

                found_path = None
                direct_candidate = crops_dir / fname
                if direct_candidate.exists():
                    found_path = direct_candidate
                else:
                    found_path = image_name_map.get(Path(fname).name)

                if found_path is None:
                    if len(missing_paths) < 10:
                        missing_paths.append(fname)
                    continue

                rel_path = found_path.relative_to(crops_dir).as_posix()
                rel_parts = Path(rel_path).parts
                document_id = rel_parts[0] if len(rel_parts) >= 3 else ''
                page_id = rel_parts[1] if len(rel_parts) >= 3 else ''
                page_fname = found_path.name

                records.append({
                    'page': {
                        'page_fname': page_fname,
                        'document_id': document_id,
                        'page_id': page_id,
                    },
                    'page_fname': page_fname,
                    'boxes': [{'transcription': gt_text}],
                    'source_line': line_no,
                })

                samples.append({
                    'page_fname': page_fname,
                    'image': str(found_path),
                    'relative_image': rel_path,
                    'document_id': document_id,
                    'page_id': page_id,
                    'box_idx': 0,
                    'gt_text': gt_text,
                })

        dataset_name = test_source.name

    else:
        raise RuntimeError(
            f'Unsupported evaluation source: {test_source}. '
            'Use either a `.json` file or a `gt.txt` file.'
        )

    return samples, records, all_images, missing_paths, dataset_name


BASELINE_MODEL_NAME = 'mlt'
BASELINE_TRAINEDDATA = find_traineddata(BASELINE_MODEL_NAME, TESSDATA_DIRS)
if BASELINE_TRAINEDDATA is None:
    raise RuntimeError(
        "Could not find `mlt.traineddata` for the baseline comparison. "
        "Install the Maltese base model first."
    )

CUSTOM_MODEL_PATH = TESSDATA_CUSTOM / f'{CFG.model_name}.traineddata'
if not CUSTOM_MODEL_PATH.exists():
    raise RuntimeError(
        f"Could not find the packaged fine-tuned model at {CUSTOM_MODEL_PATH}. "
        "Run the packaging cell in Section 5 first."
    )

BASELINE_TESSDATA_DIR = BASELINE_TRAINEDDATA.parent.resolve()
CUSTOM_TESSDATA_DIR = TESSDATA_CUSTOM.resolve()

# Evaluation source can be either a v11-style JSON file or a simple `gt.txt` file.
# TEST_SOURCE = FT_ROOT / 'mlt_data' / 'ocr_data_v11_test.json'
# CROPS_DIR = FT_ROOT / 'mlt_data' / 'crops' / 'test'
TEST_SOURCE = LOCAL_DATA / 'test' / 'gt.txt'
CROPS_DIR = LOCAL_DATA / 'test' / 'images'

eval_samples, records, all_images, missing_paths, dataset_name = load_eval_samples(TEST_SOURCE, CROPS_DIR)
rows = []

for sample in eval_samples:
    found_path = Path(sample['image'])
    gt_text = sample['gt_text']

    try:
        with Image.open(found_path) as image:
            base_pred_text = ocr_with_model(image, BASELINE_MODEL_NAME, BASELINE_TESSDATA_DIR)
            finetuned_pred_text = ocr_with_model(image, CFG.model_name, CUSTOM_TESSDATA_DIR)
    except Exception:
        base_pred_text = ''
        finetuned_pred_text = ''

    base_pred_text = normalize_eval_text(base_pred_text)
    finetuned_pred_text = normalize_eval_text(finetuned_pred_text)

    base_errors = levenshtein_distance(base_pred_text, gt_text)
    finetuned_errors = levenshtein_distance(finetuned_pred_text, gt_text)
    gt_len = len(gt_text)
    base_cer = base_errors / gt_len if gt_len else pd.NA
    finetuned_cer = finetuned_errors / gt_len if gt_len else pd.NA

    if finetuned_errors < base_errors:
        better_model = CFG.model_name
    elif base_errors < finetuned_errors:
        better_model = BASELINE_MODEL_NAME
    else:
        better_model = 'tie'

    rows.append({
        **sample,
        'base_pred_text': base_pred_text,
        'finetuned_pred_text': finetuned_pred_text,
        'base_char_errors': base_errors,
        'finetuned_char_errors': finetuned_errors,
        'gt_len': gt_len,
        'base_cer': base_cer,
        'finetuned_cer': finetuned_cer,
        'cer_delta': (finetuned_cer - base_cer) if gt_len else pd.NA,
        'better_model': better_model,
    })

df_eval_compare = pd.DataFrame(rows)
summary = {
    'dataset': dataset_name,
    'evaluation_source': str(TEST_SOURCE),
    'baseline_model': BASELINE_MODEL_NAME,
    'fine_tuned_model': CFG.model_name,
    'records_in_json': len(records),
    'crop_images_found': len(all_images),
    'matched_samples': int(len(df_eval_compare)),
    'baseline_weighted_cer': None,
    'baseline_mean_cer': None,
    'fine_tuned_weighted_cer': None,
    'fine_tuned_mean_cer': None,
    'fine_tuned_minus_base_cer': None,
    'baseline_micro_cer': None,
    'fine_tuned_micro_cer': None,
    'overall_micro_cer': None,
}

if not df_eval_compare.empty:
    total_gt_len = max(int(df_eval_compare['gt_len'].sum()), 1)
    base_weighted_cer = df_eval_compare['base_char_errors'].sum() / total_gt_len
    finetuned_weighted_cer = df_eval_compare['finetuned_char_errors'].sum() / total_gt_len
    base_mean_cer = df_eval_compare['base_cer'].dropna().mean()
    finetuned_mean_cer = df_eval_compare['finetuned_cer'].dropna().mean()

    summary['baseline_weighted_cer'] = base_weighted_cer
    summary['baseline_mean_cer'] = base_mean_cer
    summary['fine_tuned_weighted_cer'] = finetuned_weighted_cer
    summary['fine_tuned_mean_cer'] = finetuned_mean_cer
    summary['fine_tuned_minus_base_cer'] = finetuned_weighted_cer - base_weighted_cer
    summary['baseline_micro_cer'] = base_weighted_cer
    summary['fine_tuned_micro_cer'] = finetuned_weighted_cer
    summary['overall_micro_cer'] = finetuned_weighted_cer

    print(f'Evaluation source: {TEST_SOURCE}')
    print(f'Records loaded: {len(records)}')
    print(f'Crop images indexed: {len(all_images)}')
    print(f'Matched samples: {len(df_eval_compare)}')
    print(f'Baseline ({BASELINE_MODEL_NAME}) weighted CER:    {base_weighted_cer:.4f}')
    print(f'Baseline ({BASELINE_MODEL_NAME}) mean CER:        {base_mean_cer:.4f}')
    print(f'Fine-tuned ({CFG.model_name}) weighted CER: {finetuned_weighted_cer:.4f}')
    print(f'Fine-tuned ({CFG.model_name}) mean CER:     {finetuned_mean_cer:.4f}')
    print(f'Delta (fine-tuned - base):                  {finetuned_weighted_cer - base_weighted_cer:+.4f}')
    display(pd.DataFrame([summary]))
else:
    print('No evaluation data found.')
    print(f'Evaluation source: {TEST_SOURCE}')
    print(f'Records loaded: {len(records)}')
    print(f'Crop images indexed: {len(all_images)}')
    if missing_paths:
        print('First unmatched expected paths:')
        for rel_path in missing_paths[:5]:
            print(f'- {rel_path}')

# Keep backwards-compatible single-model frames for downstream cells.
if not df_eval_compare.empty:
    common_cols = ['page_fname', 'image', 'relative_image', 'document_id', 'page_id', 'box_idx', 'gt_text', 'gt_len']
    df_eval_base = df_eval_compare[common_cols + ['base_pred_text', 'base_char_errors', 'base_cer']].rename(
        columns={
            'base_pred_text': 'pred_text',
            'base_char_errors': 'char_errors',
            'base_cer': 'cer',
        }
    )
    df_eval = df_eval_compare[common_cols + ['finetuned_pred_text', 'finetuned_char_errors', 'finetuned_cer']].rename(
        columns={
            'finetuned_pred_text': 'pred_text',
            'finetuned_char_errors': 'char_errors',
            'finetuned_cer': 'cer',
        }
    )
    df_finetune_eval = df_eval.copy()
else:
    df_eval_base = pd.DataFrame()
    df_eval = pd.DataFrame()
    df_finetune_eval = pd.DataFrame()

df_eval_compare.head()

In [ ]:
# Summarize the fair v11 comparison and inspect where the fine-tuned model helps or hurts.
SNIPPET_LEN = 220

if 'df_eval_compare' not in globals() or df_eval_compare is None or df_eval_compare.empty:
    raise RuntimeError('Run the first evaluation cell in Section 6 first.')


def _preview_text(value, limit=SNIPPET_LEN):
    text = '' if value is None else str(value).replace('\n', ' ').strip()
    return text[:limit]


comparison_counts = (
    df_eval_compare['better_model']
    .value_counts(dropna=False)
    .rename_axis('better_model')
    .reset_index(name='num_samples')
)

page_compare = (
    df_eval_compare.groupby('page_fname', dropna=False)
    .agg(
        num_boxes=('box_idx', 'count'),
        base_char_errors=('base_char_errors', 'sum'),
        fine_tuned_char_errors=('finetuned_char_errors', 'sum'),
        total_gt_len=('gt_len', 'sum'),
    )
    .reset_index()
)
page_compare['base_micro_cer'] = page_compare['base_char_errors'] / page_compare['total_gt_len'].clip(lower=1)
page_compare['fine_tuned_micro_cer'] = page_compare['fine_tuned_char_errors'] / page_compare['total_gt_len'].clip(lower=1)
page_compare['cer_delta'] = page_compare['fine_tuned_micro_cer'] - page_compare['base_micro_cer']

comparison_examples = df_eval_compare.assign(
    gt_preview=lambda frame: frame['gt_text'].map(_preview_text),
    base_preview=lambda frame: frame['base_pred_text'].map(_preview_text),
    fine_tuned_preview=lambda frame: frame['finetuned_pred_text'].map(_preview_text),
)

best_improvements = comparison_examples.sort_values(
    ['cer_delta', 'finetuned_char_errors', 'page_fname', 'box_idx'],
    ascending=[True, True, True, True],
    kind='stable',
).head(10)

worst_regressions = comparison_examples.sort_values(
    ['cer_delta', 'finetuned_char_errors', 'page_fname', 'box_idx'],
    ascending=[False, False, True, True],
    kind='stable',
).head(10)

print('Per-sample wins on the exact same v11 crops:')
display(comparison_counts)

print('\nPages where the fine-tuned model improved the most:')
display(
    page_compare.sort_values(['cer_delta', 'fine_tuned_micro_cer', 'page_fname'], ascending=[True, True, True], kind='stable')
    .head(10)[['page_fname', 'num_boxes', 'base_micro_cer', 'fine_tuned_micro_cer', 'cer_delta']]
)

print('\nPages where the fine-tuned model regressed the most:')
display(
    page_compare.sort_values(['cer_delta', 'fine_tuned_micro_cer', 'page_fname'], ascending=[False, False, True], kind='stable')
    .head(10)[['page_fname', 'num_boxes', 'base_micro_cer', 'fine_tuned_micro_cer', 'cer_delta']]
)

print('\nLargest single-crop improvements:')
display(
    best_improvements[[
        'page_fname', 'box_idx', 'image', 'base_cer', 'finetuned_cer', 'cer_delta',
        'gt_preview', 'base_preview', 'fine_tuned_preview'
    ]]
)

print('Largest single-crop regressions:')
display(
    worst_regressions[[
        'page_fname', 'box_idx', 'image', 'base_cer', 'finetuned_cer', 'cer_delta',
        'gt_preview', 'base_preview', 'fine_tuned_preview'
    ]]
)

In [ ]:
# Debug: inspect the first few resolved evaluation rows.
preview_rows = []

if 'df_eval_compare' in globals() and df_eval_compare is not None and not df_eval_compare.empty:
    preview_rows = (
        df_eval_compare[['page_fname', 'relative_image', 'image']]
        .head(5)
        .assign(
            first_expected_crop=lambda frame: frame['relative_image'],
            exists=lambda frame: frame['image'].map(lambda p: Path(p).exists()),
        )
        .to_dict('records')
    )
else:
    for rec in records[:5]:
        if not isinstance(rec, dict):
            continue
        page = rec.get('page', {}) if isinstance(rec.get('page'), dict) else {}
        page_fname = str(page.get('page_fname') or rec.get('page_fname') or '').strip()
        document_id = str(page.get('document_id') or '').strip()
        page_id = str(page.get('page_id') or '').strip()
        stem = Path(page_fname).stem if page_fname else ''
        expected_rel = f'{document_id}/{page_id}/{stem}_box0000.jpg' if stem and document_id and page_id else page_fname
        preview_rows.append({
            'page_fname': page_fname,
            'document_id': document_id,
            'page_id': page_id,
            'first_expected_crop': expected_rel,
            'exists': (CROPS_DIR / expected_rel).exists() if expected_rel else False,
        })

pd.DataFrame(preview_rows)

In [ ]:
# Save the apples-to-apples v11 comparison outputs in an Excel-friendly format
OUT_DIR = FT_ROOT / 'Testing' / 'outputs'
OUT_DIR.mkdir(parents=True, exist_ok=True)

compare_out = OUT_DIR / f'{CFG.model_name}_vs_mlt_v11_eval.tsv'
summary_out = OUT_DIR / f'{CFG.model_name}_vs_mlt_v11_summary.tsv'

if 'df_eval_compare' not in globals() or df_eval_compare is None or df_eval_compare.empty:
    print('No comparison rows to save yet. Run the evaluation cells in Section 6 first.')
else:
    df_eval_compare.to_csv(compare_out, sep='\t', index=False)
    pd.DataFrame([summary]).to_csv(summary_out, sep='\t', index=False)
    print('Saved:')
    print(compare_out)
    print(summary_out)

In [ ]:
# Quick sanity check on one available evaluation crop (no hardcoded machine path).
if 'df_eval_compare' not in globals() or df_eval_compare is None or df_eval_compare.empty:
    raise RuntimeError('Run Section 6 evaluation first to populate df_eval_compare.')

sample_image = next((Path(p) for p in df_eval_compare['image'] if Path(p).exists()), None)
if sample_image is None:
    raise RuntimeError('No valid sample image found in df_eval_compare for sanity check.')

with Image.open(sample_image) as image:
    base_pred_text = ocr_with_model(image, BASELINE_MODEL_NAME, BASELINE_TESSDATA_DIR)
    finetuned_pred_text = ocr_with_model(image, CFG.model_name, CUSTOM_TESSDATA_DIR)

print(f'Sanity-check image: {sample_image}')
print(f'Baseline ({BASELINE_MODEL_NAME}) prediction: {base_pred_text}')
print(f'Fine-tuned ({CFG.model_name}) prediction: {finetuned_pred_text}')

total_gt_len_local = max(int(df_eval_compare['gt_len'].sum()), 1)
base_weighted_cer = df_eval_compare['base_char_errors'].sum() / total_gt_len_local
finetuned_weighted_cer = df_eval_compare['finetuned_char_errors'].sum() / total_gt_len_local

print(f'Baseline ({BASELINE_MODEL_NAME}) weighted CER:    {base_weighted_cer:.4f}')
print(f'Fine-tuned ({CFG.model_name}) weighted CER: {finetuned_weighted_cer:.4f}')

## 7. Worst CER Summary
Report the worst page from the v11 evaluation set and the worst single cropped sample from the FineTuning test split. Run the evaluation cells in Section 6 first.

In [ ]:
# Worst CER outputs for the apples-to-apples v11 comparison
SNIPPET_LEN = 220

if 'df_eval_compare' not in globals() or df_eval_compare is None or df_eval_compare.empty:
    raise RuntimeError('Run the evaluation cells in Section 6 first.')


def _preview_text(value, limit=SNIPPET_LEN):
    text = '' if value is None else str(value).replace('\n', ' ').strip()
    return text[:limit]


v11_page_summary = (
    df_eval_compare.groupby('page_fname', dropna=False)
    .agg(
        num_boxes=('box_idx', 'count'),
        base_char_errors=('base_char_errors', 'sum'),
        fine_tuned_char_errors=('finetuned_char_errors', 'sum'),
        total_gt_len=('gt_len', 'sum'),
    )
    .reset_index()
)
v11_page_summary['base_micro_cer'] = v11_page_summary['base_char_errors'] / v11_page_summary['total_gt_len'].clip(lower=1)
v11_page_summary['fine_tuned_micro_cer'] = v11_page_summary['fine_tuned_char_errors'] / v11_page_summary['total_gt_len'].clip(lower=1)
v11_page_summary['cer_delta'] = v11_page_summary['fine_tuned_micro_cer'] - v11_page_summary['base_micro_cer']
v11_page_summary = v11_page_summary.sort_values(
    ['fine_tuned_micro_cer', 'cer_delta', 'page_fname'],
    ascending=[False, False, True],
    kind='stable',
).reset_index(drop=True)

worst_v11_page = str(v11_page_summary.loc[0, 'page_fname'])
worst_v11_page_summary = v11_page_summary.head(1).copy()
worst_v11_page_rows = (
    df_eval_compare[df_eval_compare['page_fname'] == worst_v11_page]
    .sort_values('box_idx', kind='stable')
    .reset_index(drop=True)
    .assign(
        gt_preview=lambda frame: frame['gt_text'].map(_preview_text),
        base_preview=lambda frame: frame['base_pred_text'].map(_preview_text),
        fine_tuned_preview=lambda frame: frame['finetuned_pred_text'].map(_preview_text),
    )
)

worst_regression = (
    df_eval_compare.assign(
        gt_preview=lambda frame: frame['gt_text'].map(_preview_text),
        base_preview=lambda frame: frame['base_pred_text'].map(_preview_text),
        fine_tuned_preview=lambda frame: frame['finetuned_pred_text'].map(_preview_text),
    )
    .sort_values(['cer_delta', 'finetuned_char_errors', 'gt_len'], ascending=[False, False, False], kind='stable')
    .reset_index(drop=True)
    .head(1)
)

print('Worst v11 page by fine-tuned weighted CER:')
display(
    worst_v11_page_summary[['page_fname', 'num_boxes', 'base_micro_cer', 'fine_tuned_micro_cer', 'cer_delta']]
)
print('\nBoxes from the worst v11 page:')
display(
    worst_v11_page_rows[[
        'page_fname', 'box_idx', 'base_cer', 'finetuned_cer', 'cer_delta',
        'gt_preview', 'base_preview', 'fine_tuned_preview'
    ]]
)

print('Largest single-crop regression (fine-tuned minus base CER):')
display(
    worst_regression[[
        'image', 'page_fname', 'box_idx', 'base_cer', 'finetuned_cer', 'cer_delta',
        'gt_preview', 'base_preview', 'fine_tuned_preview'
    ]]
)